# Global Analysis of All Reconstructed Matrices

In [1]:
from pathlib import Path
import warnings
import networkx as nx
import json
import numpy as np
import pandas as pd
import torch

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


# Step 1: Load Full Reconstructed Dataset (Local PC)
Load reconstructed matrices.

In [2]:
WINDOW_LENGTH = 724
STRIDE = 10
DATASET_NAME = f'data_00_20_w{WINDOW_LENGTH}_s{STRIDE}'
RUN_NAME = 'linearAE_12dim_0005'
FILE_NAME = f'all_reconstructed_{RUN_NAME}.pt'

project_root = Path.cwd().resolve().parent
base_dir = project_root / 'models' / 'linearAE' / f'{RUN_NAME}' / 'analysis_outputs'

ALL_RECONSTRUCTED_MATRIX_FILE = base_dir / FILE_NAME
RESULTS_FOLDER = project_root / 'results' / DATASET_NAME / 'reconstruction_analysis' / RUN_NAME
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

if not ALL_RECONSTRUCTED_MATRIX_FILE.exists():
    raise FileNotFoundError(
        f"File '{ALL_RECONSTRUCTED_MATRIX_FILE.name}' not found in: {base_dir.absolute()}"
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'All Reconstructed Matrix: {ALL_RECONSTRUCTED_MATRIX_FILE.name}')

Dataset selected: data_00_20_w724_s10
All Reconstructed Matrix: all_reconstructed_linearAE_12dim_0005.pt


In [3]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')
    corr_tensor = payload.get('corr_tensor', None)
    recon_tensor = payload.get('corr_tensor_reconstructed', None)
    indices = payload.get('indices', None)
    meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
    if recon_tensor is None:
        raise KeyError('corr_tensor_reconstructed key not found in .pt file')
    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')
    if recon_tensor.ndim != 3 or recon_tensor.shape[1] != recon_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor_reconstructed: {recon_tensor.shape}')

    return corr_tensor.float(), recon_tensor.float(), indices, meta


gt_corr, recon_corr, indices, meta = load_corr_payload(ALL_RECONSTRUCTED_MATRIX_FILE)

print(f'Ground truth correlation tensor shape: {gt_corr.shape}')
print(f'Reconstructed correlation tensor shape: {recon_corr.shape}')
print(f'Indices: {indices[:10]}')
print(f'Metadata keys: {list(meta.keys())}')
print(gt_corr[:1,:5,:5])
print(recon_corr[:1,:5,:5])

num_windows, num_assets, _ = gt_corr.shape
print(f'\nNumber of windows: {num_windows}, Number of assets: {num_assets}')


Ground truth correlation tensor shape: torch.Size([41, 362, 362])
Reconstructed correlation tensor shape: torch.Size([41, 362, 362])
Indices: [152, 154, 255, 367, 350, 330, 342, 184, 144, 360]
Metadata keys: ['indices', 'split', 'meta', 'corr_tensor_reconstructed', 'tickers']
tensor([[[1.0000, 0.3398, 0.3781, 0.3090, 0.5604],
         [0.3398, 1.0000, 0.2909, 0.5142, 0.4363],
         [0.3781, 0.2909, 1.0000, 0.2982, 0.3681],
         [0.3090, 0.5142, 0.2982, 1.0000, 0.4061],
         [0.5604, 0.4363, 0.3681, 0.4061, 1.0000]]])
tensor([[[1.1970, 0.3195, 0.2980, 0.3277, 0.5094],
         [0.3226, 1.1682, 0.2761, 0.4140, 0.4044],
         [0.3054, 0.2649, 1.2388, 0.2845, 0.3582],
         [0.2908, 0.4392, 0.3094, 1.2548, 0.3868],
         [0.5180, 0.4100, 0.3572, 0.3901, 1.2541]]])

Number of windows: 41, Number of assets: 362


# Errors Statistics (MSE,MAE,Frobenius on full Reconstructed Dataset)

In [4]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [5]:
gt_corr_np = gt_corr.cpu().numpy()
recon_corr_np = recon_corr.cpu().numpy()

errors_df, errors_stats = reconstruction_errors(gt_corr_np, recon_corr_np)

print('Reconstruction error statistics (Full Dataset - no matrix sanitization):')
display(errors_stats)
# 7. Preparazione payload JSON
file_path = RESULTS_FOLDER / 'reconstruction_errors.json'
# Inverti righe e colonne con .T
errors_stats_dict = errors_stats.T.to_dict()

Reconstruction error statistics (Full Dataset - no matrix sanitization):


,mean,std,min,median,max
MSE,0.006385,0.002017,0.003948,0.005548,0.011074
MAE,0.061464,0.009437,0.048349,0.058578,0.081751
Frobenius,28.593060,4.432686,22.745716,26.964638,38.094219


In [6]:
def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}

def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    # Edges overlap and Jaccard
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    # Degree distribution and L1 distance
    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    # Average path length (weighted by distance)
    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    # Average path length (unweighted, treating all edges as length 1)
    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    # Betweenness centrality top-k overlap
    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [7]:
def sanitize_correlation_matrix(corr_matrix):
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    return corr_matrix

def build_distance_matrix(corr_matrix):
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
    np.fill_diagonal(dist_matrix, 0.0)
    
    return dist_matrix

def corr_to_mst(corr_matrix):
    """
    Converte una matrice di correlazione in un oggetto MST di NetworkX.
    Usa la metrica di distanza d = sqrt(2 * (1 - rho))
    """
    # 1. Calcolo della matrice delle distanze
    corr_matrix = sanitize_correlation_matrix(corr_matrix)
    dist_matrix = build_distance_matrix(corr_matrix)
    
    # 2. Creazione del grafo completo
    G = nx.from_numpy_array(dist_matrix)
    
    # 3. Calcolo del Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G, weight='weight')
    
    return mst

In [8]:
# Inizializziamo una lista per contenere i risultati di ogni coppia di matrici
all_metrics = []

n_samples = gt_corr_np.shape[0]
n_assets = gt_corr_np.shape[1]
top_k_centrality = 10

print(f"Inizio elaborazione di {n_samples} matrici...")

for i in range(n_samples):
    # 1. Estrazione della singola matrice (2D)
    curr_gt = gt_corr_np[i]
    curr_recon = recon_corr_np[i]
    
    # 2. Generazione degli MST per questa istanza
    mst_gt = corr_to_mst(curr_gt)
    mst_recon = corr_to_mst(curr_recon)
    
    # 3. Calcolo metriche
    metrics = compare_mst_metrics(
        mst_gt, 
        mst_recon, 
        n_assets=n_assets, 
        top_k=top_k_centrality
    )
    
    # Aggiungiamo alla lista
    all_metrics.append(metrics)
    
    # Feedback ogni 50 iterazioni per monitorare il progresso
    if (i + 1) % 10 == 0:
        print(f"Elaborate {i + 1}/{n_samples} matrici...")

# 4. Creazione DataFrame e calcolo della media
results_df = pd.DataFrame(all_metrics)

# 5. Calcolo delle statistiche aggregate
stats_df = results_df.describe().T 

# 6. Conversione in un dizionario strutturato
# 'index' orient crea un dizionario dove le chiavi sono le metriche
stats_dict = stats_df.to_dict(orient='index')

# 7. Salvataggio unico di tutte le metriche
combined_metrics = {
    'reconstruction_errors': errors_stats_dict,
    'mst_metrics': stats_dict,
}

with open(file_path, 'w') as f:
    json.dump(combined_metrics, f, indent=4)

print(f"Statistiche salvate con successo in: {file_path}")

# --- Visualizzazione rapida a schermo delle statistiche aggregate ---
print("\n--- Summary Statistics (Full Dataset) ---")
display(stats_df)

Inizio elaborazione di 41 matrici...
Elaborate 10/41 matrici...
Elaborate 20/41 matrici...
Elaborate 30/41 matrici...
Elaborate 40/41 matrici...
Statistiche salvate con successo in: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results\data_00_20_w724_s10\reconstruction_analysis\linearAE_12dim_0005\reconstruction_errors.json

--- Summary Statistics (Full Dataset) ---


,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,41.0,38.423080,4.129767,29.085873,36.565097,39.335180,41.274238,46.537396
edge_jaccard_pct,41.0,23.857740,3.116937,17.017828,22.372881,24.482759,26.003490,30.324910
degree_l1,41.0,0.128689,0.045479,0.038674,0.099448,0.121547,0.160221,0.226519
avg_path_len,41.0,7.960094,1.610006,5.443772,6.801897,7.580208,8.993270,11.561340
avg_path_len_recon,41.0,9.092559,2.658465,5.335355,6.505792,9.646057,11.356281,14.289968
avg_path_len_diff,41.0,1.783801,1.271223,0.067012,0.747993,1.703470,2.576715,5.129079
avg_path_len_unweighted,41.0,10.610209,1.846656,7.997291,9.401953,10.005816,11.590349,14.683491
avg_path_len_unweighted_recon,41.0,10.903918,1.036953,9.456911,10.181479,10.574004,11.692827,13.844722
avg_path_len_unweighted_diff,41.0,1.550513,1.043125,0.070094,0.733154,1.453253,2.091030,4.485499
betweenness_topk_overlap_pct,41.0,26.097561,10.217154,10.000000,20.000000,20.000000,30.000000,50.000000
